In [1]:
# 변수의 shape, type, value 확인
def p(var,_name="") :
    if _name != "" : print(f'<<{_name}>>')
    if type(var)!=type([]):
        try:
            print(f'Shape:{var.shape}')
        except :
            pass
    print(f'Type: {type(var)}')
    print(f'Values: {var}')

def pst(_x,_name=""):
    print(f'<<{_name}>> Shape{_x.shape}, {type(_x)}')
def ps(_x,_name=""):
    print(f'<<{_name}>> Shape{_x.shape}')

# LangChain RAG(Retrieval-Augmented Generation) Agent 구현

In [2]:
%%time
#!pip install -q --upgrade --force-reinstall langchain langchain-openai langchain-community faiss-cpu sentence-transformers hnswlib
!pip install langchain-openai langchain-community faiss-cpu sentence-transformers hnswlib -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
CPU times: user 1.49 s, sys: 259 ms, total: 1.75 s
Wall time: 53.6 s


In [3]:
import langchain
import langchain_community
# import langchain_classic # Remove this import to avoid potential conflicts if not strictly needed

langchain.__version__, langchain_community.__version__
#('1.2.4', '0.4.1') 2601

('1.2.4', '0.4.1')

In [4]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [5]:
import os
from google.colab import userdata

# OpenAI API 키는 필요시 설정 (이번 실습에서는 사용하지 않음)
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

## AG용 샘플 매뉴얼 파일 생성

In [6]:
%%writefile semiconductor_manual.txt
# 반도체 공정 기술 매뉴얼 v3.0

## 1. 서론
이 문서는 최신 반도체 제조 공정에 대한 기술적 세부사항을 다룹니다. 각 공정 단계는 수율과 성능에 직접적인 영향을 미치므로, 모든 파라미터는 엄격하게 관리되어야 합니다.

## 2. 증착 공정 (Deposition)
증착은 웨이퍼 위에 얇은 막(Thin Film)을 형성하는 과정입니다. 화학 기상 증착(CVD)과 물리 기상 증착(PVD)으로 나뉩니다. CVD는 가스의 화학 반응을 이용하며, PVD는 플라즈마를 이용하여 물리적으로 박막을 증착시킵니다.

## 3. 포토리소그래피 (Photolithography)
포토리소그래피, 또는 노광 공정은 웨이퍼에 회로 패턴을 새기는 핵심 단계입니다. 감광액(PR)이 도포된 웨이퍼에 마스크를 통과한 빛을 쬐어 패턴을 형성합니다. 중요 파라미터는 빛의 파장, 노광량(Dose), 그리고 초점(Focus)입니다. 파장이 짧을수록 더 미세한 회로를 만들 수 있습니다.

## 4. 식각 공정 (Etching)
식각은 불필요한 부분을 선택적으로 제거하여 회로 패턴을 완성하는 과정입니다. 습식 식각(Wet Etching)과 건식 식각(Dry Etching)이 있습니다. 건식 식각은 플라즈마를 사용하여 미세하고 수직적인 패턴을 만드는 데 유리하여 현재 주류로 사용됩니다.

## 5. 화학적 기계적 연마 (CMP)
CMP(Chemical Mechanical Polishing)는 웨이퍼 표면을 화학적, 기계적 방법을 통해 거울처럼 평탄하게 만드는 공정입니다. 다음 공정의 안정성을 위해 표면의 단차를 제거하는 것이 매우 중요하며, 슬러리의 종류와 연마 압력이 주요 변수입니다.



Writing semiconductor_manual.txt


## 임베딩 모델 로드 및 문서 분할

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

### 사전 훈련된 임베딩 모델 로드
# 'all-MiniLM-L6-v2'는 384차원의 벡터를 생성하며, 영어/다국어 환경에서 좋은 성능을 보입니다.
# Sentence-BERT (SBERT) 계열로, 문장 단위를 임베딩 하도록 훈련된 모델
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("임베딩 모델 로드 완료.")

# 문서 로드 및 청크(chunk) 분할
# 여기서는 간단하게 문단 단위("\n\n")로 분할합니다.
with open('semiconductor_manual.txt','r',encoding='utf-8') as f:
    docs = f.read().split("\n\n")

print(f"문서가 총 {len(docs)}개의 청크로 분할되었습니다.")
print("--- 첫 번째 청크 내용 ---")
print(docs[0])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

임베딩 모델 로드 완료.
문서가 총 7개의 청크로 분할되었습니다.
--- 첫 번째 청크 내용 ---
# 반도체 공정 기술 매뉴얼 v3.0


## 문서 임베딩 및 FAISS 인덱스 생성/저장

In [ ]:
import faiss

# 모든 문서 청크를 인코딩하여 벡터로 변환
print("문서 임베딩 생성 중...")
embeddings = embed_model.encode(docs, convert_to_numpy=True)
print(f"임베딩 생성 완료. 임베딩 행렬 크기: {embeddings.shape}") # (문서 청크 수, 임베딩 차원)

# FAISS 인덱스 생성
# IndexFlatL2는 L2 거리(유클리드 거리)를 사용하여 모든 벡터와 직접 비교하는 가장 기본적인 인덱스입니다.
index_faiss = faiss.IndexFlatL2(embeddings.shape[1])

# 인덱스에 임베딩 데이터 추가
index_faiss.add(embeddings)
print(f"FAISS 인덱스에 {index_faiss.ntotal}개의 벡터가 추가되었습니다.")

# 인덱스 파일 저장 (재사용을 위해)
faiss.write_index(index_faiss, 'manual_faiss.index')
print("'manual_faiss.index' 파일이 저장되었습니다.")

문서 임베딩 생성 중...
임베딩 생성 완료. 임베딩 행렬 크기: (7, 384)
FAISS 인덱스에 7개의 벡터가 추가되었습니다.
'manual_faiss.index' 파일이 저장되었습니다.


# Retrieval 함수 및 RAG 파이프라인 통합
사용자의 질문을 받아 관련 문서를 검색하고, 이 정보를 바탕으로 LLM이 답변을 생성하는 RAG 파이프라인을 구축합니다.


## FAISS 기반 검색(Retrieval) 함수
>질문을 벡터로 변환한 뒤, FAISS 인덱스에서 가장 유사한 문서 청크를 찾아내는 함수

In [ ]:
# FAISS 인덱스에서 쿼리와 가장 관련 높은 k개의 문서를 검색
def retrieve_from_faiss(query: str, k: int = 2) -> list[str]:
    # 저장된 인덱스 로드
    index = faiss.read_index('manual_faiss.index')

    # 쿼리를 벡터로 변환
    query_vector = embed_model.encode([query], convert_to_numpy=True)

    # 인덱스 검색 (D: 거리, I: 인덱스 번호)
    distances, indices = index.search(query_vector, k)

    # 검색된 인덱스 번호에 해당하는 원본 문서 청크 반환
    return [docs[i] for i in indices[0]]

In [ ]:
# --- 함수 테스트 ---
test_query = "포토리소그래피 공정의 중요 파라미터는 뭐야?"
retrieved_docs = retrieve_from_faiss(test_query, k=2)

print(f"[질문]: {test_query}")
print("\n[검색된 문서]:")
for i, doc in enumerate(retrieved_docs):
    print(f"--- 문서 {i+1} ---")
    print(doc)

[질문]: 포토리소그래피 공정의 중요 파라미터는 뭐야?

[검색된 문서]:
--- 문서 1 ---
## 3. 포토리소그래피 (Photolithography)
포토리소그래피, 또는 노광 공정은 웨이퍼에 회로 패턴을 새기는 핵심 단계입니다. 감광액(PR)이 도포된 웨이퍼에 마스크를 통과한 빛을 쬐어 패턴을 형성합니다. 중요 파라미터는 빛의 파장, 노광량(Dose), 그리고 초점(Focus)입니다. 파장이 짧을수록 더 미세한 회로를 만들 수 있습니다.
--- 문서 2 ---
## 1. 서론
이 문서는 최신 반도체 제조 공정에 대한 기술적 세부사항을 다룹니다. 각 공정 단계는 수율과 성능에 직접적인 영향을 미치므로, 모든 파라미터는 엄격하게 관리되어야 합니다.


## RAG 파이프라인 구현 및 실행
>검색 -> 보강 -> 생성의 3단계 파이프라인을 구현하고 실행



In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# LLM 초기화 (Generator 역할)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 프롬프트 템플릿 정의
# LLM에게 검색된 컨텍스트를 기반으로 답변하도록 명확하게 지시합니다.
template = """당신은 반도체 공정 전문가입니다. 주어진 컨텍스트 정보를 바탕으로 사용자의 질문에 답변해주세요.
컨텍스트에 질문과 관련없는 내용이 있다면, 컨텍스트를 무시하고 아는 대로 답변하세요.

컨텍스트:
{context}

질문:
{question}

답변:
"""
prompt = ChatPromptTemplate.from_template(template)

# RAG 체인(파이프라인) 정의
# LCEL(LangChain Expression Language)을 사용하여 파이프라인을 구성
rag_chain = (
    {"context": (lambda x: "\n---\n".join(retrieve_from_faiss(x["question"]))),
     "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
#  파이프라인 실행
user_question = "포토리소그래피 공정이란 무엇이고, 왜 중요한가요?"
print(f"[사용자 질문]: {user_question}\n")

print("[RAG 파이프라인 답변]:")
# invoke 메소드의 입력은 체인의 첫 번째 컴포넌트가 기대하는 형태와 일치해야 합니다.
# 여기서는 lambda 함수의 입력 x, 즉 {"question": user_question} 입니다.
response = rag_chain.invoke({"question": user_question})
print(response)

[사용자 질문]: 포토리소그래피 공정이란 무엇이고, 왜 중요한가요?

[RAG 파이프라인 답변]:
포토리소그래피 공정은 반도체 제조에서 회로 패턴을 웨이퍼에 전사하는 과정입니다. 이 공정은 감광제(포토레지스트)를 사용하여 특정 영역에 빛을 비추고, 그 결과로 생성된 패턴을 기반으로 후속 공정에서 회로를 형성합니다.

포토리소그래피는 반도체 제조에서 매우 중요한 단계입니다. 그 이유는 다음과 같습니다:

1. **정밀한 패턴 형성**: 포토리소그래피는 나노미터 단위의 정밀한 패턴을 형성할 수 있어, 고집적 회로(IC)의 성능과 밀도를 높이는 데 기여합니다.

2. **수율 향상**: 정확한 패턴 전사는 전체 제조 공정의 수율에 큰 영향을 미칩니다. 잘못된 패턴은 결함을 초래하고, 이는 최종 제품의 품질 저하로 이어질 수 있습니다.

3. **기술 발전**: 포토리소그래피 기술은 지속적으로 발전하고 있으며, 더 작은 공정 노드(예: 7nm, 5nm 등)를 지원하기 위해 새로운 기술(예: 극자외선(EUV) 리소그래피)이 개발되고 있습니다. 이는 반도체 산업의 경쟁력을 유지하는 데 필수적입니다.

결론적으로, 포토리소그래피 공정은 반도체 제조의 핵심 요소로, 고성능 및 고수율의 반도체 소자를 생산하는 데 필수적입니다.


## HNSW 적용해 보기

In [ ]:
import hnswlib
import time

# HNSWLib 인덱스 생성 및 저장
print("HNSWLib 인덱스 생성 중...")
dim = embeddings.shape[1]
num_elements = embeddings.shape[0]

index_hnsw = hnswlib.Index(space='l2', dim=dim) # L2(유클리드) 거리 사용
index_hnsw.init_index(max_elements=num_elements, ef_construction=200, M=16)
index_hnsw.add_items(embeddings, np.arange(num_elements))
index_hnsw.set_ef(50)
index_hnsw.save_index("manual_hnsw.index")
print("'manual_hnsw.index' 파일이 저장되었습니다.")

# HNSWLib 기반 검색 함수 구현
def retrieve_from_hnsw(query: str, k: int = 2) -> list[str]:
    """HNSWLib 인덱스에서 쿼리와 가장 관련 높은 k개의 문서를 검색합니다."""
    index = hnswlib.Index(space='l2', dim=dim)
    index.load_index("manual_hnsw.index", max_elements=num_elements)

    query_vector = embed_model.encode([query])
    labels, distances = index.knn_query(query_vector, k=k)

    return [docs[i] for i in labels[0]]

HNSWLib 인덱스 생성 중...
'manual_hnsw.index' 파일이 저장되었습니다.


In [ ]:
# 성능 비교 실험---
print("\n--- 검색 성능 비교 ---")
comparison_query = "CMP 공정은 무엇을 하는 단계인가?"
print(f"[비교 질문]: {comparison_query}\n")

# FAISS 속도 측정
start_time = time.time()
faiss_results = retrieve_from_faiss(comparison_query, k=1)
faiss_time = time.time() - start_time
print(f"FAISS 검색 시간: {faiss_time:.6f}초")
print(f"FAISS 결과: {faiss_results[0][:100]}...")

# HNSWLib 속도 측정
start_time = time.time()
hnsw_results = retrieve_from_hnsw(comparison_query, k=1)
hnsw_time = time.time() - start_time
print(f"\nHNSWLib 검색 시간: {hnsw_time:.6f}초")
print(f"HNSWLib 결과: {hnsw_results[0][:100]}...")

# 데이터가 적을 경우 속도 차이가 미미하거나, 인덱스 로딩 시간 때문에 오히려 FAISS가 빠를 수도 있음


--- 검색 성능 비교 ---
[비교 질문]: CMP 공정은 무엇을 하는 단계인가?

FAISS 검색 시간: 0.011229초
FAISS 결과: ## 1. 서론
이 문서는 최신 반도체 제조 공정에 대한 기술적 세부사항을 다룹니다. 각 공정 단계는 수율과 성능에 직접적인 영향을 미치므로, 모든 파라미터는 엄격하게 관리되어야 ...

HNSWLib 검색 시간: 0.006393초
HNSWLib 결과: ## 1. 서론
이 문서는 최신 반도체 제조 공정에 대한 기술적 세부사항을 다룹니다. 각 공정 단계는 수율과 성능에 직접적인 영향을 미치므로, 모든 파라미터는 엄격하게 관리되어야 ...


In [ ]:
#  HNSWLib를 RAG 파이프라인에 통합하여 최종 답변 생성
user_question = "포토리소그래피 공정이란 무엇이고, 왜 중요한가요?"
print(f"[사용자 질문]: {user_question}\n")

print("[RAG 파이프라인 답변]:")
print("\n--- HNSWLib 기반 RAG 파이프라인 실행 ---")
rag_chain_hnsw = (
    {"context": (lambda x: "\n---\n".join(retrieve_from_hnsw(x["question"]))), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
#response_hnsw = rag_chain_hnsw.invoke({"question": comparison_query})
response_hnsw = rag_chain_hnsw.invoke({"question": user_question})
print(response_hnsw)

[사용자 질문]: 포토리소그래피 공정이란 무엇이고, 왜 중요한가요?

[RAG 파이프라인 답변]:

--- HNSWLib 기반 RAG 파이프라인 실행 ---
포토리소그래피 공정은 반도체 제조에서 회로 패턴을 웨이퍼에 전사하는 과정입니다. 이 공정은 감광제(포토레지스트)를 웨이퍼 표면에 도포한 후, 특정 파장의 빛을 사용하여 원하는 패턴을 형성하는 방식으로 진행됩니다. 빛이 비춰진 부분의 감광제가 화학적으로 변화하여, 이후 현상 과정을 통해 패턴이 남게 됩니다.

포토리소그래피는 반도체 소자의 미세한 구조를 형성하는 데 필수적이며, 소자의 성능과 수율에 직접적인 영향을 미칩니다. 따라서 이 공정의 정확성과 정밀성은 반도체 제조의 성공에 매우 중요합니다. 고해상도 패턴을 형성할 수 있는 능력은 반도체의 집적도와 성능을 결정짓는 핵심 요소 중 하나입니다.


# << Query Augment >>

In [ ]:
# 하나의 질문을 여러 관점으로 변환
original_query = "반도체 제조 과정에서 웨이퍼 처리 방법은?"

# LLM이 여러 버전의 질문 생성
generated_queries = [
    "웨이퍼 가공 기술에 대해 알려주세요",
    "반도체 웨이퍼의 처리 단계는 무엇인가요?",
    "실리콘 웨이퍼 제조 공정을 설명해주세요" ]

In [ ]:
# 원본 질문을 의미적으로 관련된 여러 키워드로 확장
query = "반도체 수율 향상"
expanded_queries = [
    "반도체 수율 향상",
    "웨이퍼 품질 개선",
    "공정 최적화",
    "defect reduction" ]

In [ ]:
# 각 query를 vector로 변환
query_vectors = [embed_model.encode(q) for q in generated_queries]

## LangChain에 있는 MultiQueryRetriever 사용하기   
**어떤방식으로 어떤 query들이 만들어지는지 관리할 수없음:도메인에 맞는 방식으로 설계 어려움**


In [ ]:
## LangChain 구현 예시

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
#from langchain_openai import ChatOpenAI # Added import for ChatOpenAI
from langchain_community.vectorstores import FAISS # Added import for LangChain's FAISS

# Convert raw document chunks to LangChain Document objects
langchain_documents = [Document(page_content=doc) for doc in docs]

# Initialize HuggingFaceEmbeddings for LangChain compatibility
hf_embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

# Create a FAISS vectorstore from the documents and embedding model
vectorstore = FAISS.from_documents(langchain_documents, hf_embeddings) # Corrected call

print("LangChain FAISS vectorstore created.")

# Multi-Query Retriever 설정
llm = ChatOpenAI(temperature=0)
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    llm=llm
)

# MultiQueryRetriever: 내부적으로 다음과 같이 동작
# 1. 원본 질문을 3-5개의 다른 표현으로 변환
# 2. 각각을 vector로 변환하여 검색
# 3. 결과를 합치고 중복 제거
query = "반도체 수율 향상 방법은?"
results = retriever.invoke(query)

# results
# [Document(id='f4de4b2d-3f97-418e-8e49-2f762d951dc8', metadata={}, page_content='## 1. 서론\n이 문서는 최신 반도체 제조 공정에 대한 기술적 세부사항을 다룹니다. 각 공정 단계는 수율과 성능에 직접적인 영향을 미치므로, 모든 파라미터는 엄격하게 관리되어야 합니다.'),
#  Document(id='2621e419-6754-4095-adbf-0cde9c27b605', metadata={}, page_content='# 반도체 공정 기술 매뉴얼 v3.0'),
#  Document(id='ec8f9327-83c9-422e-88af-4756e4c4b502', metadata={}, page_content='## 4. 식각 공정 (Etching)\n식각은 불필요한 부분을 선택적으로 제거하여 회로 패턴을 완성하는 과정입니다. 습식 식각(Wet Etching)과 건식 식각(Dry Etching)이 있습니다. 건식 식각은 플라즈마를 사용하여 미세하고 수직적인 패턴을 만드는 데 유리하여 현재 주류로 사용됩니다.')]

In [ ]:
## System prompt와 Retrieve된 관련문서를 포함하여 Prompt 생성

context = "\n\n".join([f"[{i}] {d.page_content}" for i, d in enumerate(results, 1)])
"""[1] ## 1. 서론\n이 문서는 최신 반도체 제조 공정에 대한 기술적 세부사항을 다룹니다.
각 공정 단계는 수율과 성능에 직접적인 영향을 미치므로, 모든 파라미터는 엄격하게 관리되어야 합니다.\n\n
[2] # 반도체 공정 기술 매뉴얼 v3.0\n\n
[3] ## 4. 식각 공정 (Etching)\n식각은 불필요한 부분을 선택적으로 제거하여 회로 패턴을 완성하는 과정입니다.
습식 식각(Wet Etching)과 건식 식각(Dry Etching)이 있습니다. 건식 식각은 플라즈마를 사용하여 미세하고 수직적인 패턴을 만드는 데 유리하여 현재 주류로 사용됩니다.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 반도체 공정 전문가입니다. 주어진 문서 근거만 사용해 답변하세요. 불확실하면 불확실하다고 말하세요."),
    ("human", "질문: {question}\n\n문서:\n{context}\n\n요청: 문서 근거로 핵심 요점 5개와 실행 가능한 개선 항목 5개를 정리해줘.")
])

answer = llm.invoke(prompt.format_messages(
    question = query,  # "반도체 수율 향상 방법은?",
    context = context # retrieve된 문서
))

print(answer.content)

핵심 요점 5개:
1. 반도체 제조 공정의 각 단계는 수율과 성능에 직접적인 영향을 미침.
2. 식각 공정은 회로 패턴을 완성하기 위해 불필요한 부분을 제거하는 과정임.
3. 습식 식각과 건식 식각이 있으며, 현재 주류는 건식 식각임.
4. 건식 식각은 플라즈마를 사용하여 미세하고 수직적인 패턴을 만드는 데 유리함.
5. 모든 파라미터는 엄격하게 관리되어야 함.

실행 가능한 개선 항목 5개:
1. 건식 식각 공정의 효율을 높이기 위해 플라즈마 조건을 최적화하고, 장비의 정확도를 개선할 수 있음.
2. 식각 공정 중 발생하는 부품의 오염을 줄이기 위해 정기적인 청소 및 유지보수를 강화할 필요가 있음.
3. 식각 공정에서 사용되는 화학약품의 품질을 관리하여 일관된 결과물을 얻을 수 있도록 해야 함.
4. 식각 공정의 온도와 압력 등의 조건을 정밀하게 제어하여 일관된 제품 품질을 유지할 필요가 있음.
5. 식각 공정에서 발생하는 잔류물을 효율적으로 제거하기 위한 후속 공정을 개선하여 수율을 높일 수 있음.


## Query Augment를 어떻게 해야 하나  

### 유사한 질문 만들기

**original_query: 반도체 제조 과정에서 웨이퍼 처리 방법은?**

A. 동의어/표현 변형 (의미 동일, 문장만 바꿈)

- 반도체 공정에서 웨이퍼는 어떤 방식으로 처리되나요?

- 웨이퍼 처리(가공) 공정은 제조 과정에서 어떻게 이루어지나요?

- 반도체 제조 라인에서 웨이퍼 핸들링/처리 절차를 설명해 주세요.

- 웨이퍼 공정 처리 플로우(흐름)는 어떻게 구성되나요?

- 웨이퍼 전처리/후처리 방식에는 무엇이 있나요?

B. 범위 확장형 (웨이퍼 처리 “방법”을 세분화)

- 웨이퍼 세정(Cleaning) 공정의 대표 방법과 목적은 무엇인가요?

- 포토리소그래피에서 웨이퍼 표면 처리(코팅/베이크)는 어떻게 진행되나요?

- 식각(Etch) 공정에서 웨이퍼는 어떤 조건으로 처리되나요?

- 증착(CVD/PVD/ALD) 공정에서 웨이퍼 처리 방식의 차이는 무엇인가요?

- CMP 공정에서 웨이퍼 평탄화 처리 원리와 주요 변수는 무엇인가요?

C. 공정/장비/품질 관점으로 변형 (RAG 문서 매칭 폭 확장)

- 웨이퍼 처리 장비(Track, Stepper, Etcher 등)별 핵심 처리 단계는 무엇인가요?

- 웨이퍼 핸들링(이송/로딩/언로딩)에서 결함을 줄이는 방법은 무엇인가요?

- 웨이퍼 공정 중 오염(파티클/금속) 관리와 처리 방법은 무엇인가요?

- 웨이퍼 열처리(Anneal/RTP) 조건이 공정 결과에 미치는 영향은 무엇인가요?

- 공정 중 웨이퍼 검사/계측(Inspection/Metrology)은 어떤 방식으로 수행되나요?

### 유사한 키워드로 확장하기

**query: 반도체 수율 향상**

A. 동의 키워드/근접 용어 확장

- 수율 개선, 수율 최적화, 수율 증대

- 불량률 감소, 결함 감소, defect reduction

- 공정 최적화, 공정 안정화, 공정 조건 최적화

- 품질 개선, 품질 안정화, 품질 관리(QC)

B. 원인/메커니즘 기반 확장 (수율을 떨어뜨리는 요인 → 개선 관점)

- 파티클 관리, 오염 제어, 금속 오염, 유기 오염

- 공정 변동성 감소, SPC(통계적 공정관리), 공정 능력(Cp/Cpk)

- 장비 드리프트, 챔버 컨디션, 유지보수(PM), 장비 매칭

- 레시피 튜닝, 공정 윈도우(process window), DOE(실험계획)

C. 결함/검사/분석 기반 확장 (수율 향상에 직접 연결되는 검색 축)

- 결함 검사(Inspection), 결함 맵(Defect map), 결함 분류(ADC)

- 웨이퍼 맵 분석, 수율 맵(yield map), 빈도/패턴 분석

- 수율 분석(yield analysis), 결함 원인 분석(RCA), FA(Failure Analysis)

- EDA(Exploratory Data Analysis), 이상탐지(Anomaly detection)EDA(탐색적 데이터 분석), 이상탐지(이상 탐지)

D. 데이터/AI/모델 기반 확장 (RAG에서 기술 문서 매칭에 유리)

- 공정 데이터 분석, 센서 데이터, FDC(Fault Detection & Classification)

- 가상계측(Virtual Metrology), Run-to-Run(R2R) 제어

- 예지보전(PdM), 공정 이상탐지, 결함 예측, 수율 예측

- 머신러닝 기반 수율 개선, 딥러닝 결함 검출, GNN 기반 공정 그래프 분석

E. 공정별 수율 향상 축 (문서가 공정 단위로 구성된 경우 특히 유리)

- 포토 수율 개선, 오버레이(overlay) 최적화, CD 균일도

- 식각 균일도 개선, 증착 균일도, CMP 디싱/에로전 감소

- 열처리 조건 최적화, 스트레스/워페이지 관리

- 패키징 수율, 테스트 수율, 번인/최종검사 수율

## Query Augment를 위한 System Prompt  

### **"RAG에 적용할 증강 쿼리를 위한 system prompt를 만들어 보자."**

당신은 검색 효율이 높은 쿼리 변형을 생성하는 데 특화된 "RAG 쿼리 증강 전문가"입니다.
당신의 임무는 사용자의 입력 쿼리를 원래 의도를 유지하면서 문서 검색 재현율과 검색 범위가 향상된 증강 쿼리 세트로 변환하는 것입니다.

## 핵심 목표
1) 의도 유지: 사용자의 실제 질문/주제를 변경하지 않습니다.

2) 검색 범위 확장: 문서에 나타날 가능성이 높은 동의어, 의역어, 도메인 관련 패싯을 추가합니다.

3) 모호성 감소: 쿼리가 불충분한 경우, 사용자에게 질문하지 않고 모호성을 해소하는 데 초점을 맞춘 변형을 생성합니다.

4) 검색 친화성: 기술 문서, 표제어 및 색인 용어와 일치하는 키워드가 풍부한 쿼리를 선호합니다.

## 출력 요구 사항
- 유효한 JSON 형식으로만 출력합니다(마크다운, 주석, 후행 쉼표 사용 금지).

- 입력이 한국어인 경우 기본 언어를 한국어로 사용합니다.

- 각 질의 문자열은 입력 문자열 자체가 120자(한국어 공백 포함)보다 길지 않은 한 120자 이하여야 합니다.

- 중복 및 유사 질의는 피하십시오.

- 입력 문자열에 없는 특정 숫자, 회사명 또는 독점 용어를 임의로 만들어내지 마십시오.

- 답변이나 설명은 포함하지 말고, 질의만 작성하십시오.

## 질의 증강 전략  
다음 범주에 걸쳐 다양한 변형을 생성합니다(해당되는 경우):  
A. 의역 paraphrase: 다른 표현이지만 의미는 동일  
B. 동의어 synonyms: 핵심 용어의 동의어 대체  
C. 패싯 확장 facet_expansion: 관련 하위 주제 추가(공정/결함/장비/데이터/지표 등)  
D. 약어 및 영어 용어 acronyms_english: 일반적인 영어 동의어 포함(예: 수율, SPC, FDC)  
E. 범위 좁히기 narrowing: 더 구체적인 버전(하위 공정, 방법, 단계)  
F. 범위 넓히기 broadening: 약간 더 넓은 범위이지만 여전히 주제 관련(상류/하류 관련)  
G. 오류 허용 error_tolerant: 일반적인 오타/띄어쓰기 변형(최소한)   

## 도메인 힌트(반도체 제조)
- 수율/의심되는 원인: 결함, 오염, 입자, 드리프트, 변동성, SPC, Cp/Cpk
- 장비/공정: 리소그래피, 에칭, 증착(CVD/PVD/ALD), CMP, 어닐링/RTP, 계측, 검사
- 데이터/AI: FDC, 가상 계측, 런투런 제어, 이상 탐지, 근본 원인 분석, 웨이퍼 맵 분석

## JSON schema
    {
      "original_query": "<string>",
      "language": "ko" | "en" | "mixed",
      "intents": ["<short intent label>", ...],
      "must_include_terms": ["<term>", ...],
      "augmented_queries": {
        "paraphrase": ["<q1>", "<q2>", ...],
        "synonyms": ["<q1>", "<q2>", ...],
        "facet_expansion": ["<q1>", "<q2>", ...],
        "acronyms_english": ["<q1>", "<q2>", ...],
        "narrowing": ["<q1>", "<q2>", ...],
        "broadening": ["<q1>", "<q2>", ...],
        "error_tolerant": ["<q1>", "<q2>", ...]
      },
      "deduped_all": ["<q1>", "<q2>", ...],
      "top_k_recommended": ["<q1>", "<q2>", ...]
    }

## 제약 조건
- 전체 증강 쿼리(deduped_all) 수는 20~45개 사이여야 합니다.
- top_k_recommended: 버킷 전반에 걸쳐 다양성을 최대화하는 8개의 쿼리를 선택합니다.

- 입력이 매우 짧은 경우(8자 이하), facet_expansion 및 acronyms_english를 강조합니다.

- 입력에 여러 주제가 포함된 경우, 두 주제 모두 유지하고 조합을 생성합니다.

이제 사용자의 다음 메시지를 입력 쿼리로 사용하여 작업을 수행합니다.

## **도메인 특화 Query Augment**
: RAG성능 향상을 위해 도메인에 특화된 정교한 Augmentation 방식을 만들어 보자  

“반도체 수율 향상”을 검색할 때는 **공정 / 결함 / 장비 / 데이터** 축을 섞어 2~3개 버전으로 분기하는 게 검색 성능이 안정적입니다.

예: 반도체 수율 향상 + 결함 원인 분석 + 웨이퍼 맵

예: 반도체 수율 개선 + SPC + 공정 변동성

예: 수율 예측 + FDC + 이상탐지

### 반도체 공정 관리 관점에서 Query Augment용 System Prompt 만들기

**반도체 공정 관리에 특화된 증강 쿼리를 만들어 주는 system prompt를 작성해줘.
반도체 공정 관리에 필요한 특성을 분류하고, 분류에 따라 어떻게 달라 질 수 있는지 검토하고, 이에 맞는 시스템 프롬프트를 작성해줘. 입력 쿼리를 판단하여 서로 다른 증강쿼리를 적용할 수 있어야 함**

### Query 특성 분류

반도체 공정 관리는 단순 제조 지식이 아니라 통제·분석·의사결정이 핵심이므로, 질의는 아래 축으로 명확히 갈립니다.

(A) 공정 단계 중심 질의
-  예: 포토 공정 관리, 식각 균일도 문제, CMP 공정 안정화
-  특징  
-- 특정 공정(step)이 명확  
-- 물리적 파라미터, 레시피, 장비 조건 중심  

(B) 품질/수율 중심 질의
- 예: 수율 저하 원인, 결함 증가 문제, 불량률 감소
- 특징  
-- 결과(yield, defect) 중심  
-- 여러 공정에 걸친 문제일 가능성 큼

(C) 통계적 공정 관리(SPC) 중심 질의

- 예: SPC 적용 방법, 관리도 설정, Cp/Cpk 해석
- 특징  
-- 통계 용어, 관리 지표 중심  
-- 공정 일반화 성향 강함  

(D) 장비/설비 상태 관리 질의

- 예: 장비 드리프트, 챔버 컨디션, PM 주기  
- 특징    
-- 시간 축, 장비 이력, 유지보수와 강하게 연결     

(E) 결함 분석 / 원인 분석(RCA) 질의
- 예: 특정 패턴 결함 원인, wafer map 해석  
- 특징  
-- Inspection / Metrology / FA 문서와 잘 매칭됨  

(F) 데이터·AI 기반 공정 관리 질의
- 예: FDC 모델, 이상 탐지, 가상 계측  
- 특징  
-- 용어가 영어 혼용  
-- 논문, 특허, 기술 보고서 매칭이 중요  

(G) 운영·의사결정 중심 질의
- 예: 공정 관리 전략, 운영 최적화, 의사결정 기준  
- 특징  
-- 기술 + 관리 문서 혼합  
-- broad query가 많음  

### 분류에 따라 증강 쿼리가 달라져야 하는 이유  
**“모든 질의에 동일한 증강 규칙을 적용하면 RAG 성능이 떨어진다”**

| 분류      | 증강 방향                 | 이유                       |
| ------- | --------------------- | ------------------------ |
| A 공정 단계 | **세분화(Narrowing)**    | 문서는 공정별로 분리되어 있음         |
| B 수율/품질 | **원인 확장(Facet)**      | 직접 원인보다 간접 원인이 문서에 많음    |
| C SPC   | **지표/용어 확장**          | Cp, Cpk, 관리도 등 키워드 매칭 중요 |
| D 장비    | **시간·상태 축 추가**        | Drift, aging, PM 용어 필수   |
| E 결함    | **분석 방법 중심 확장**       | RCA, pattern, map 문서가 많음 |
| F AI    | **영문/약어 확장**          | FDC, VM, R2R 등  모델링·추론·자동화 기법 |
| G 운영    | **범위 확장(Broadening)** | 전략·가이드 문서 탐색 목적          |


### 입력 쿼리 판단 로직 (자동 분류 기준)
**LLM이 입력 쿼리를 보고 내부적으로 판단해야 할 규칙**

1 키워드 기반 1차 분류

- 공정명 포함 → (A)

- 수율, 불량, defect → (B)

- SPC, 관리도, Cp → (C)

- 장비, 챔버, PM → (D)

- 검사, 맵, 패턴 → (E)

- 모델, 예측, 탐지 → (F)

- 전략, 관리, 운영 → (G)

2 복수 분류 허용

- 예: “수율 향상을 위한 FDC 적용”
→ (B + F)

3 분류 결과에 따라 증강 버킷 가중치 변경

- (A): narrowing 40%, facet 30%

- (B): facet 40%, broadening 20%

- (F): acronyms/english 40%

### 분류별 증강 전략 요약
(A) 공정 단계 중심
- 공정명 + 세부 파라미터
- 공정명 + 균일도 / 변동성
- 공정명 + 장비 조건

(B) 수율/품질 중심
- 수율 + 원인 + 공정
- 수율 + 결함 + 분석
- 수율 + 관리 방법

(C) SPC 중심
- SPC + 적용
- 관리도 + 해석
- Cp/Cpk + 기준

(D) 장비 중심
- 장비 드리프트 + 영향
- 챔버 상태 + 공정 결과
- PM + 수율

(E) 결함 분석
- 결함 패턴 + 원인
- wafer map + 분석
- inspection + RCA검사 + RCA

(F) AI/데이터
- FDC + anomaly detectionFDC + 이상 탐지
- virtual metrology + 예측
- run-to-run + 제어

(G) 운영/전략
- 공정 관리 전략
- 공정 안정화 방법
- 운영 최적화

## 반도체 공정 관리 전용 증강 쿼리 생성 System Prompt 예시


```
당신은 "반도체 공정 제어 쿼리 증강 전문가"입니다.
당신의 임무는 반도체 공정 제어, 수율 관리 및 제조 분석에 특화된 RAG 시스템을 위한 검색 최적화 증강 쿼리를 생성하는 것입니다.

## 핵심 원칙
 항상 원래 의도를 유지합니다.
 공정 사양, SPC 매뉴얼, 수율 보고서,
장비 로그, 검사 지침 및 기술 문서와 같은 내부 문서에 최적화합니다.

## 1단계. 쿼리 분류 (내부 추론만 해당)
입력 쿼리를 다음 범주 중 하나 이상으로 분류합니다.  
 A) 공정 단계 중심  
 B) 수율/품질 중심  
 C) 통계적 공정 제어(SPC)  
 D) 장비/툴 상태  
 E) 결함 분석/근본 원인 분석(RCA)  
 F) 데이터/AI 기반 공정 제어  
 G) 운영/전략  
복수 범주를 선택할 수 있습니다.

## 2단계. 적응형 데이터 증강 전략
분류 유형에 따라 다음과 같은 증강 가중치를 적용합니다.
 A: 분류 세분화 및 공정 매개변수 확장 강조
 B: 근본 원인 및 공정 간 연관성 확장 강조
 C: 지표, 관리도, 통계 용어 강조
 D: 시간, 드리프트, 유지보수, 장비 상태 강조
 E: 검사, 웨이퍼 맵, 결함 패턴 분석 강조
 F: 영어 용어, 약어, 모델 기반 쿼리 강조
 G: 분류 범위 확장 및 관리 지향적 언어 강조

## JSON schema
  {
    "original_query": "<string>",
    "detected_categories": ["A","B","F"],
    "augmented_queries": {
      "narrowing": [],
      "facet_expansion": [],
      "acronyms_english": [],
      "broadening": []
    },
    "deduped_all": [],
    "top_k_recommended": []
  }

## 출력 규칙
 유효한 JSON 형식으로만 출력합니다.
 설명을 포함하지 마십시오.
 질문에 대한 답변을 포함하지 마십시오.
 해당 도메인에서 영어 용어가 표준으로 사용되는 경우가 아니면 한국어를 사용하십시오.
 중복 및 유사 중복 항목을 피하십시오.
```

### System prompt 적용 결과 예시

```
{
  "original_query": "반도체 수율이 갑자기 떨어졌을 때 어떤 데이터를 먼저 봐야 하나?",
  "detected_categories": ["B", "E", "C", "D"],
  "augmented_queries": {
    "narrowing": [
      "수율 급락 발생 시 1차 확인 데이터 목록",
      "수율 하락 원인 파악을 위한 우선순위 지표",
      "라인에서 수율 급락 시 바로 보는 검사/계측 항목",
      "Lot 단위 수율 급락 트러블슈팅 체크리스트",
      "수율 급락 시 공정별 영향 범위 확인 방법"
    ],
    "facet_expansion": [
      "수율 급락 원인 분석 SPC 이상 신호 연계",
      "수율 하락과 결함(Defect) 증가 상관 분석",
      "수율 급락과 공정 변동성(variation) 원인",
      "수율 하락 시 공정 조건 변경 이력 추적",
      "수율 급락과 장비 드리프트/챔버 컨디션 점검",
      "수율 급락 시 wafer map 패턴 기반 원인 분류",
      "수율 저하와 파티클/오염(contamination) 영향",
      "수율 하락 시 공정 윈도우(process window) 이탈 점검",
      "수율 급락 시 메트롤로지(CD/Overlay) 이상 확인",
      "수율 저하와 공정 레시피 변경/버전 관리"
    ],
    "acronyms_english": [
      "yield drop troubleshooting data checklist",
      "yield excursion SPC signals control chart",
      "defect map analysis for yield loss",
      "RCA for sudden yield drop in fab",
      "equipment drift impact on yield excursion",
      "Cp Cpk shift and yield degradation"
    ],
    "broadening": [
      "수율 이슈 대응 프로세스 운영 가이드",
      "수율 이상 탐지 및 원인 분석 표준 절차",
      "수율 리포트(yield report) 구성과 주요 지표",
      "결함 분석(FA/RCA) 프로세스와 데이터 흐름"
    ]
  },
  "deduped_all": [
    "수율 급락 발생 시 1차 확인 데이터 목록",
    "수율 하락 원인 파악을 위한 우선순위 지표",
    "라인에서 수율 급락 시 바로 보는 검사/계측 항목",
    "Lot 단위 수율 급락 트러블슈팅 체크리스트",
    "수율 급락 시 공정별 영향 범위 확인 방법",
    "수율 급락 원인 분석 SPC 이상 신호 연계",
    "수율 하락과 결함(Defect) 증가 상관 분석",
    "수율 급락과 공정 변동성(variation) 원인",
    "수율 하락 시 공정 조건 변경 이력 추적",
    "수율 급락과 장비 드리프트/챔버 컨디션 점검",
    "수율 급락 시 wafer map 패턴 기반 원인 분류",
    "수율 저하와 파티클/오염(contamination) 영향",
    "수율 하락 시 공정 윈도우(process window) 이탈 점검",
    "수율 급락 시 메트롤로지(CD/Overlay) 이상 확인",
    "수율 저하와 공정 레시피 변경/버전 관리",
    "yield drop troubleshooting data checklist",
    "yield excursion SPC signals control chart",
    "defect map analysis for yield loss",
    "RCA for sudden yield drop in fab",
    "equipment drift impact on yield excursion",
    "Cp Cpk shift and yield degradation",
    "수율 이슈 대응 프로세스 운영 가이드",
    "수율 이상 탐지 및 원인 분석 표준 절차",
    "수율 리포트(yield report) 구성과 주요 지표",
    "결함 분석(FA/RCA) 프로세스와 데이터 흐름"
  ],
  "top_k_recommended": [
    "수율 급락 발생 시 1차 확인 데이터 목록",
    "수율 급락과 장비 드리프트/챔버 컨디션 점검",
    "수율 급락 시 wafer map 패턴 기반 원인 분류",
    "수율 급락 원인 분석 SPC 이상 신호 연계",
    "수율 급락 시 메트롤로지(CD/Overlay) 이상 확인",
    "yield excursion SPC signals control chart",
    "defect map analysis for yield loss",
    "결함 분석(FA/RCA) 프로세스와 데이터 흐름"
  ]
}
```

    {
      "original_query": "식각 공정에서 균일도가 흔들릴 때 원인과 관리 포인트는?",
      "detected_categories": ["A", "D", "C", "E"],
      "augmented_queries": {
        "narrowing": [
          "식각 균일도(uniformity) 저하 주요 원인",
          "식각 공정 균일도 변동 관리 포인트",
          "식각 균일도 흔들림 트러블슈팅 절차",
          "식각 레시피 조건 변화가 균일도에 미치는 영향",
          "식각 균일도와 챔버 상태(컨디션) 점검 항목",
          "식각 공정 가스/압력/RF 파워 변동과 균일도",
          "식각 공정 온도 제어 불안정과 균일도",
          "식각 공정 엔드포인트(Endpoint) 변동과 균일도"
        ],
        "facet_expansion": [
          "식각 균일도와 장비 드리프트 원인 분석",
          "식각 균일도 SPC 관리도 설정 및 경보 기준",
          "식각 균일도와 파티클/오염 영향",
          "식각 균일도 이상 시 wafer map 패턴 해석",
          "식각 균일도와 챔버 클리닝/PM 주기 최적화",
          "식각 균일도 변동과 로트 간/웨이퍼 간 분산 분석",
          "식각 공정 조건 DOE로 균일도 최적화",
          "식각 균일도와 CD 변화 상관 분석",
          "식각 균일도와 가스 유량(MFC) 이상 징후"
        ],
        "acronyms_english": [
          "etch uniformity troubleshooting",
          "etch chamber condition drift uniformity",
          "SPC control chart for etch uniformity",
          "endpoint detection variation etch uniformity",
          "PM cycle optimization for etch tool",
          "MFC drift impact on etch uniformity"
        ],
        "broadening": [
          "식각 공정 안정화 방법 및 운영 가이드",
          "공정 균일도 관리 지표와 표준 절차",
          "장비 상태 기반 공정 관리(best practices)"
        ]
      },
      "deduped_all": [
        "식각 균일도(uniformity) 저하 주요 원인",
        "식각 공정 균일도 변동 관리 포인트",
        "식각 균일도 흔들림 트러블슈팅 절차",
        "식각 레시피 조건 변화가 균일도에 미치는 영향",
        "식각 균일도와 챔버 상태(컨디션) 점검 항목",
        "식각 공정 가스/압력/RF 파워 변동과 균일도",
        "식각 공정 온도 제어 불안정과 균일도",
        "식각 공정 엔드포인트(Endpoint) 변동과 균일도",
        "식각 균일도와 장비 드리프트 원인 분석",
        "식각 균일도 SPC 관리도 설정 및 경보 기준",
        "식각 균일도와 파티클/오염 영향",
        "식각 균일도 이상 시 wafer map 패턴 해석",
        "식각 균일도와 챔버 클리닝/PM 주기 최적화",
        "식각 균일도 변동과 로트 간/웨이퍼 간 분산 분석",
        "식각 공정 조건 DOE로 균일도 최적화",
        "식각 균일도와 CD 변화 상관 분석",
        "식각 균일도와 가스 유량(MFC) 이상 징후",
        "etch uniformity troubleshooting",
        "etch chamber condition drift uniformity",
        "SPC control chart for etch uniformity",
        "endpoint detection variation etch uniformity",
        "PM cycle optimization for etch tool",
        "MFC drift impact on etch uniformity",
        "식각 공정 안정화 방법 및 운영 가이드",
        "공정 균일도 관리 지표와 표준 절차",
        "장비 상태 기반 공정 관리(best practices)"
      ],
      "top_k_recommended": [
        "식각 균일도 흔들림 트러블슈팅 절차",
        "식각 균일도와 챔버 상태(컨디션) 점검 항목",
        "식각 균일도 SPC 관리도 설정 및 경보 기준",
        "식각 균일도와 챔버 클리닝/PM 주기 최적화",
        "식각 균일도 이상 시 wafer map 패턴 해석",
        "etch uniformity troubleshooting",
        "SPC control chart for etch uniformity",
        "MFC drift impact on etch uniformity"
      ]
    }


    {
      "original_query": "FDC로 공정 이상을 탐지하려면 어떤 피처를 쓰는 게 좋은가?",
      "detected_categories": ["F", "D", "C"],
      "augmented_queries": {
        "narrowing": [
          "FDC 공정 이상 탐지에 유용한 센서 피처 설계",
          "FDC 피처 엔지니어링 기준(통계/주파수/추세)",
          "챔버 센서 로그 기반 FDC 피처 추천",
          "공정 단계별(FDC) 피처 선택 가이드",
          "FDC 입력 피처와 라벨 정의 방법"
        ],
        "facet_expansion": [
          "FDC 이상 탐지 피처와 장비 드리프트 구분",
          "FDC 피처와 SPC 지표 연계 방법",
          "FDC에서 알람 기준(threshold) 설정 방법",
          "FDC 피처 중요도 분석(Feature importance) 사례",
          "FDC에서 시계열 윈도우링/요약 통계 피처",
          "FDC에서 주파수 도메인 피처(FFT) 적용",
          "FDC 피처와 결함/수율(yield) 상관 분석",
          "FDC 모델 학습을 위한 데이터 전처리(누락/동기화)",
          "FDC에서 공정 레시피/상태 변수 포함 여부"
        ],
        "acronyms_english": [
          "FDC feature engineering for anomaly detection",
          "sensor log features for fault detection and classification",
          "drift vs fault detection features in FDC",
          "SPC and FDC feature integration",
          "time-series window features for FDC",
          "threshold setting for FDC alarms"
        ],
        "broadening": [
          "FDC 시스템 구축 절차와 운영 가이드",
          "공정 이상 탐지 모델 비교(FDC vs SPC)",
          "가상계측(Virtual Metrology)과 FDC 연계"
        ]
      },
      "deduped_all": [
        "FDC 공정 이상 탐지에 유용한 센서 피처 설계",
        "FDC 피처 엔지니어링 기준(통계/주파수/추세)",
        "챔버 센서 로그 기반 FDC 피처 추천",
        "공정 단계별(FDC) 피처 선택 가이드",
        "FDC 입력 피처와 라벨 정의 방법",
        "FDC 이상 탐지 피처와 장비 드리프트 구분",
        "FDC 피처와 SPC 지표 연계 방법",
        "FDC에서 알람 기준(threshold) 설정 방법",
        "FDC 피처 중요도 분석(Feature importance) 사례",
        "FDC에서 시계열 윈도우링/요약 통계 피처",
        "FDC에서 주파수 도메인 피처(FFT) 적용",
        "FDC 피처와 결함/수율(yield) 상관 분석",
        "FDC 모델 학습을 위한 데이터 전처리(누락/동기화)",
        "FDC에서 공정 레시피/상태 변수 포함 여부",
        "FDC feature engineering for anomaly detection",
        "sensor log features for fault detection and classification",
        "drift vs fault detection features in FDC",
        "SPC and FDC feature integration",
        "time-series window features for FDC",
        "threshold setting for FDC alarms",
        "FDC 시스템 구축 절차와 운영 가이드",
        "공정 이상 탐지 모델 비교(FDC vs SPC)",
        "가상계측(Virtual Metrology)과 FDC 연계"
      ],
      "top_k_recommended": [
        "FDC 피처 엔지니어링 기준(통계/주파수/추세)",
        "FDC 이상 탐지 피처와 장비 드리프트 구분",
        "FDC에서 시계열 윈도우링/요약 통계 피처",
        "FDC에서 알람 기준(threshold) 설정 방법",
        "FDC 피처와 결함/수율(yield) 상관 분석",
        "FDC feature engineering for anomaly detection",
        "sensor log features for fault detection and classification",
        "SPC and FDC feature integration"
      ]
    }


## 동적 RAG Augment System prompt 적용의 문제점  
**입력된 쿼리를 보고 동적으로 System Prompt를 만들어서 적용하면?**  

### 장점 : 왜 “동적 system prompt 생성”이 매력적인가

- 입력 쿼리가 넓고(공정/장비/SPC/결함/AI/운영 등) 문서도 이질적이라,
- 증강 전략을 잘못 고르면 검색품질이 아주 낮아짐
- 따라서 입력을 보고 그 순간 최적의 증강 전략을 선택하는 건 RAG 품질에 직접적으로 이득

### 단점 : 현실적으로 발생하는 문제
- 재현성 붕괴: 같은 질문인데 답이 달라져서 운영 이슈/감사 대응이 어려움
- 프롬프트 인젝션 표면 확대: “프롬프트를 생성하는 프롬프트”는 공격면이 늘어남
- 평가 난이도 폭증: 템플릿이 매번 바뀌면 A/B와 회귀 테스트가 사실상 불가능
- 문서 거버넌스 충돌: 어느 문서군을 우선 탐색해야 하는지 정책이 흔들리고, 사용자는 “검색이 불안정”하다고 느낌

### 현실적 대안 : “생성”이 아니라 “선택+조립”으로  
1) 고정 메타 System Prompt (항상 동일)
- 역할: 분류→정책 적용→출력 JSON 형식→금지사항→토큰/길이 제한을 강제
- 여기에는 “반도체 공정 관리 공통 규칙”만 둡니다.

2) 템플릿 라이브러리(도메인/문서군별)
예:
- SPC/관리도/능력지수 템플릿
- 장비 드리프트/PM/챔버 컨디션 템플릿
- 결함/wafer map/RCA/FA 템플릿
- 공정(포토/식각/증착/CMP/열처리) 레시피/윈도우 템플릿
- FDC/VM/R2R/이상탐지(ML) 템플릿
- 운영/의사결정/라인 정책 템플릿

3) 라우터(입력 쿼리 분류기)
- 입력 쿼리를 A ~ G 다중라벨로 분류하고, 상위 1 ~ 3개 템플릿을 선택
- 선택 결과를 “시스템 프롬프트 생성”이 아니라 템플릿 ID + 슬롯 값으로 표현

4) 동적 요소는 “슬롯”만 주입
- 예: must_include_terms, acronyms, facet 후보, 공정명/장비명 추출, 우선 버킷 가중치
- 즉 “프롬프트를 새로 쓰지 말고”, 정해진 프롬프트에 값만 넣기

# Query Augmented RAG 적용 흐름(Agent관점)

1. User Query 수신
2. Classifier/Router: (A ~ G) 라벨 + confidence분류기/라우터: (A ~ G) 라벨 + 신뢰도
3. Template Selector: 템플릿 1~3개 선택
4. Augmenter LLM: 선택 템플릿에 슬롯 주입 → 증강쿼리 JSON 생성
5. Retriever: hybrid(BM25+벡터)로 검색
6. Answer agent: 검색 결과 기반 답변

추가로 운영 안정성을 위해:  
(a) 캐시: (query signature, template ids, slots) 기준으로 증강 결과 캐싱  
(b) 정책 게이트: 너무 broad하거나 금지 키워드(사내 민감정보) 포함 시 정제 규칙 적용  
(c) 회귀 테스트: 대표 질의 세트에 대해 top-k recall/precision 모니터링  

## 단일 vector로 한번만 Query하기:결합검색 Merged-Query Retrieval (vs. Multi-Query Retrieval)

### 단일 Vector가 유리한 경우  
1. “단일 vector가 유리한 경우”의 본질적 정의  
단일 vector 검색이 유리하려면, 다음 3가지가 동시에 성립해야 합니다.
- 쿼리가 **단일 의미 축**을 가진다
- 검색 대상 문서가 의미적으로 동질적이다
- **“부분적으로만 맞는 문서”가 쓸모가 없다**

  이 조건을 만족하지 못하면, 단일 vector는 거의 항상 손해입니다.

2. 단일 vector가 유리한 대표적 경우들  
- (A) 매우 **구체적**이고 기술적으로 수렴된 질의
  - 의미가 하나로 강하게 수렴
  - 결합해도 의미 희석이 거의 없음
  - 오히려 개별 검색은 중복 결과만 늘림

- (B) FAQ / 매뉴얼 / 가이드 문서 검색(부분적으로 맞는 문서의 가치 낮음)
  - broad하지만 문서군이 단일
  - 문서 구조(목차/제목)가 유사
  - multi-query는 과잉 탐색

- (C) 요약·개요·전략 수준 질의(교육/보고용)
  - 여러 의미를 섞는 게 목적
  - 단일 중심 벡터가 오히려 적합

- (D) 문서 자체가 이미 “통합 관점”으로 작성된 경우
  - 문서가 공정+장비+통계를 한꺼번에 다룸
  - 백서, 기술 보고서, 컨설팅 자료
  - 이 경우 multi-query로 쪼개면, 같은 문서를 여러 번 가져오는 부작용 발생

### Semantic Dilution: 대부부의 경우 단일 Vector가 불리
1 증강 쿼리를 결합하면 무슨 일이 생기는가

여러 의미를 가진 쿼리를 결합하면:  
 vector = 공정(process) + 장비(equipment) + SPC(statistics) + 결함 (defect) + AI (FDC)  
--> 이 서로 멀리 떨어진 의미 축들이 하나의 vector로 압축  

결과: **결국 모두에게 애매한 벡터**가 됨

In [ ]:
### 단일 vector로 변환 방법들 ###

# 1) 각각 검색 후 합치기
def multi_vector_search(query_vectors, vectorstore, k=5):
    all_results = []
    # 각 query vector로 개별 검색
    for query_vec in query_vectors:
        results = vectorstore.similarity_search_by_vector(query_vec, k=k)
        all_results.extend(results)
    # 중복 제거 및 점수 합산
    unique_results = remove_duplicates_and_merge_scores(all_results)
    return unique_results[:k]

# 2) Vector 평균화
def average_vector_search(query_vectors, vectorstore, k=5):
    # 여러 query vector의 평균 계산
    avg_vector = np.mean(query_vectors, axis=0)
    # 평균 vector로 검색
    results = vectorstore.similarity_search_by_vector(avg_vector, k=k)
    return results

# 3) 가중 평균
def weighted_vector_search(query_vectors, weights, vectorstore, k=5):
    # 가중 평균 계산 (중요한 query에 더 큰 가중치)
    weighted_avg = np.average(query_vectors, weights=weights, axis=0)

    results = vectorstore.similarity_search_by_vector(weighted_avg, k=k)
    return results

# 4) 최대값 방식 (Max Similarity)
def max_similarity_search(query_vectors, vectorstore, k=5):
    chunk_scores = {}
    # 각 chunk에 대해 모든 query vector와의 유사도 중 최댓값 사용
    for query_vec in query_vectors:
        results_with_scores = vectorstore.similarity_search_with_score_by_vector(query_vec, k=50)
        for chunk, score in results_with_scores:
            chunk_id = chunk.page_content
            if chunk_id not in chunk_scores:
                chunk_scores[chunk_id] = score
            else:
                chunk_scores[chunk_id] = max(chunk_scores[chunk_id], score)
    # 점수 순으로 정렬
    sorted_chunks = sorted(chunk_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_chunks[:k]

# 5) 앙상블 방식
def ensemble_search(query_vectors, vectorstore, k=5):
    method_results = []
    # 여러 방법으로 검색
    method_results.append(average_vector_search(query_vectors, vectorstore, k))
    method_results.append(max_similarity_search(query_vectors, vectorstore, k))
    #  ...
    # 결과 투표/가중 합산
    final_results = combine_results_by_voting(method_results)
    return final_results[:k]


# << Chunking >>


## 청킹 방식 정리
1. **고정 길이** 기반 청킹 (Fixed-size Chunking) : 256 / 512 / 1024  토큰 수 또는 문자 수 기준으로 문서를 분할  
- 의미 단위가 깨질 가능성 큼  
- 구조가 약한 텍스트(**대량 로그**, OCR 결과)에 낮은 비용으로 적용

2. **오버랩** 기반 청킹 (Overlapping Chunking) : 고정 길이(512) + 일부 토큰을 겹치게(64 ~ 128:10 ~ 25%) 분할
- 검색 recall 향상
- 벡터 중복, 저장/검색 비용 증가

3. **문단(Paragraph)** 기반 청킹 : 줄바꿈, 공백, 문단 구분 기준으로 분할
- **문단 길이 편차** 큼, 최소/최대 토큰 제한과 결합(100~500 tokens)

4. **문장(Sentence)** 기반 청킹 : 문장 단위로 나눈 뒤 여러 문장을 묶음(NLTK, spaCy, KoNLPy 등 활용)
- 의미 보존 최상, 정밀 질의(QA)에 유리
- 짧은 문장 위주의 문서에서는 비효율

5. **구조 인식** 청킹 (Structure-aware Chunking) : 문서의 논리 구조(제목, 소제목, 섹션, 리스트, 표 등)를 참조
- 질문–답변 정합성 매우 높음(**기술 문서, 매뉴얼에 최적**)
- 전처리 복잡하고, 문서 포맷 의존적

6. **의미** 기반 청킹 (Semantic Chunking) : 문장단위 임베딩 유사도를 기준으로 자동 분할
- 문장 단위 임베딩, 문장 간 similarity 계산, similarity 급변 지점에서 분할
- 의미 단위 보존 최상, **RAG 품질 최고 수준**
- 비용 큼 (RAG 품질을 극대화해야 하는 경우)

7. **하이브리드** 청킹 (Hybrid Chunking) : 여러 전략을 조합
- 구조 기반 → 문단 기반 → 토큰 제한
- 문장 기반 → 의미 기반 병합
- 고정 길이 + semantic boundary 보정
- 실무에서 가장 안정적, **다양한 질의 유형** 대응 가능

### 청킹은 단순 전처리가 아니라 RAG 설계의 일부
일반적인 추천 순서
1) 구조 기반 가능 여부 확인
2) 문단/문장 기반 + 토큰 제한
3) 고급 시스템에서는 semantic/hybrid 적용

## LLM을 사용한 Chunking   
최근 1~2년 사이 “유효한 선택지”로 자리 잡은 방식  
- LLM context window: 128k ~ 1M tokens 현실화
- 입력 토큰 비용 하락
- RAG의 복잡성(청킹·임베딩·검색 튜닝)에 대한 피로

문서 수가 적고, 해당 문서가 **지식 자산**에 가까운 경우
(논문, 내부 기술 문서, 리서치 노트)라면  
Full-document → LLM 청킹 → 고정 저장 방식은  
 **실용 단계**에 들어왔음  

필수 조건
- 문서 수가 적음: 대략 수십 ~ 수백 개
- 문서 길이가 김 : 수천~수만 token
- 오프라인 처리
- 원문 변형 허용(또는 변형이 있는지 추가 감사 필요)
- 요약등 재서술, 구조화가 반드시 필요한 경우(규제, 법률, 의료등은 위험)

## Chunking을 위한 System prompt

```
당신은 문서를 검색 기반 LLM 시스템(RAG)에 적합하도록
의미 단위로 분할하는 전처리 전문가이다.

다음 원문 문서를 읽고, 아래 조건을 모두 만족하도록 chunk로 분할하라.

[청킹 목적]
 각 chunk는 검색 시 독립적으로 사용 가능해야 한다.
 질문–답변(QA), 요약, 설명 요청에 자연스럽게 대응할 수 있어야 한다.

[청킹 기준]
1. 각 chunk는 하나의 중심 주제 또는 논리적 단위를 가져야 한다.
2. 문맥상 강하게 연결된 문장은 같은 chunk에 포함하라.
3. 주제가 명확히 전환되는 지점에서는 새로운 chunk를 시작하라.
4. 목록, 정의, 절차, 설명은 가능한 한 분리하지 말고 함께 유지하라.
5. 불필요한 요약, 재서술, 정보 추가는 절대 하지 말고 원문 표현을 그대로 유지하라.

[크기 제약]
 각 chunk의 길이는 최소 {{MIN_TOKENS}} tokens 이상,
  최대 {{MAX_TOKENS}} tokens 이하여야 한다.
 단, 의미적으로 불가피한 경우에만 예외를 허용하라.

[출력 형식]
 출력은 JSON 배열이어야 한다.
 각 chunk는 다음 필드를 포함해야 한다.
  {
    "chunk_id": 정수 (0부터 시작),  
    "title": 해당 chunk의 핵심 주제를 한 문장으로 요약한 제목,  
    "content": 원문에서 추출한 실제 텍스트,  
    "start_context": chunk가 시작되는 문장의 핵심 요약 (1문장),  
    "end_context": chunk가 끝나는 문장의 핵심 요약 (1문장)  
  }

[중요한 제약]
 원문에 없는 내용을 생성하지 말 것.
 문장의 순서를 변경하지 말 것.
 chunk 간 내용이 중복되지 않도록 할 것.
 모든 원문 내용은 정확히 하나의 chunk에만 포함되어야 한다.
 chunk 분할 판단 시 개인적 선호나 창의적 판단을 배제하고, 동일 문서에 대해 항상 유사한 경계를 선택하라.(재현성 강화)
 title과 context 요약은 반드시 content에 포함된 정보만 사용하라.(Hallucination 방지)

이제 아래에 주어진 문서를 처리하라.
```

## 문서 유형 인식 + 적응형 청킹 System Prompt  

RAG 품질 문제의 상당수는 “청킹 알고리즘”이 아니라
문서 유형을 무시한 설계 부재에서 발생함.
- 분류 결과가 틀렸을 때 자동 보정 전략은?
- chunk 품질을 정량 점수로 평가하는 기준은?

```
당신은 검색 기반 LLM 시스템(RAG)을 위한 문서 분석 및 청킹 전처리 전문가이다.

주어진 문서를 먼저 분석하여 문서의 종류와 구조적 특성을 분류한 뒤,
해당 유형에 가장 적합한 청킹 전략을 선택하여 문서를 분할하라.

[1단계: 문서 유형 분류]  
먼저 문서를 읽고 아래 항목을 분류하라.
1. 문서 유형(document_type)
  아래 중 하나 또는 가장 가까운 것으로 판단하라.
   "technical"        : 기술 문서, API 설명, 시스템 설계
   "academic"         : 논문, 연구 보고서
   "legal_policy"     : 법률, 규정, 약관, 내부 규칙
   "manual_guide"     : 매뉴얼, 사용 가이드, 절차서
   "report_analysis"  : 분석 리포트, 시장/경영 보고서
   "narrative"        : 설명문, 서술형 문서, 에세이
   "mixed"            : 복합 구조 문서

2. 구조적 특성(structure_features)
  해당되는 항목을 모두 선택하라.
   명확한 섹션/제목 존재
   문단 중심
   문장 단위 설명 위주
   목록(list), 표, 정의(definition) 다수
   절차(step-by-step) 구조
   주제 전환이 잦음
   주제 응집도가 높음

[2단계: 청킹 전략 선택]  
위 분류 결과를 바탕으로 다음 원칙을 따른다.

 technical / academic / legal_policy
  → 구조 인식 기반 청킹을 우선 적용
  → 섹션 또는 논리 블록 단위 유지

 manual_guide
  → 절차·목록 단위 보존
  → 단계가 끊기지 않도록 청킹

 report_analysis
  → 주제 단위(소주제) 중심 청킹
  → 배경–분석–결론을 분리하지 말 것

 narrative
  → 문단/문장 기반 의미 청킹
  → 주제 전환 지점에서 분리

 mixed
  → 문단 기반 청킹 후 의미적으로 강하게 연결된 단위를 병합

[3단계: 공통 청킹 규칙]  
 각 chunk는 검색 시 독립적으로 사용 가능해야 한다.
 하나의 중심 주제 또는 질문에 대응 가능한 단위여야 한다.
 주제가 명확히 바뀌는 지점에서만 새로운 chunk를 시작하라.
 정의, 목록, 절차, 예시는 가능한 한 분리하지 말고 함께 유지하라.
 요약, 재서술, 정보 추가는 절대 하지 말고 원문을 그대로 사용하라.
 문서의 모든 내용은 정확히 하나의 chunk에만 포함되어야 한다.

[4단계: 크기 제약]  
 각 chunk는 최소 {{MIN_TOKENS}} tokens 이상,
  최대 {{MAX_TOKENS}} tokens 이하를 권장한다.
 단, 법률·정의·절차 문서는 의미 보존을 위해
  최대 토큰을 약간 초과할 수 있다.

[5단계: 출력 형식]  
출력은 반드시 JSON 객체 하나로 구성하라.
  {
    "document_classification": {
      "document_type": "...",
      "structure_features": ["...", "..."],
      "selected_chunking_strategy": "..."
    },
    "chunks": [
      {
        "chunk_id": 0,
        "title": "해당 chunk의 핵심 주제를 나타내는 제목 (1문장)",
        "content": "원문에서 그대로 추출한 텍스트",
        "start_context": "이 chunk가 시작되는 부분의 핵심 의미 (1문장)",
        "end_context": "이 chunk가 끝나는 부분의 핵심 의미 (1문장)"
      }
    ]
  }

[중요한 제약]  
 원문에 없는 내용을 생성하지 말 것.
 문장 순서 변경 금지.
 chunk 간 내용 중복 금지.
 분류 결과와 실제 청킹 전략이 일관되게 유지되도록 할 것.

이제 아래 문서를 처리하라.
```

# DB 저장

In [ ]:
## RDBMS에 저장하기
for i, doc in enumerate(docs):
    json_data = {
        "id": f"chunk_{i}",
        "content": doc,  # 원본 콘텐츠
        "embedding_index": i,  # FAISS 인덱스에서의 위치
        "start_context": "시작 문맥 요약",
        "end_context": "끝 문맥 요약",
        "metadata": {
            "timestamp": "2026-01-25T12:36:00Z",
            "source": "문서 제목",
            "tags": ["tag1", "tag2"]
        }
    }
    db.insert(json_data)